# Bioinformatics pipeline (DADA2 and VSEARCH)

This notebook implements DADA2 and VSEARCH _de novo_ pipelines for amplicon data, covering all steps through to taxonomy assignment.

The pipeline includes:
- Removal of Ns and primers (cutadapt)
- Read quality control, filtering, and trimming (filterAndTrim)
- Error modelling, dereplication, denoising, read merging and chimera removal (DADA2 & VSEARCH)

The DADA2 workflow is adapted from the [NEOF version](https://github.com/khmaher/HPC_dada2/tree/main?tab=readme-ov-file), and the VSEARCH workflow is based on a published [GitHub pipeline](https://github.com/torognes/vsearch) and [VSEARCH documentation](https://torognes.github.io/vsearch/), both modified for this project.

Taxonomic assignment is performed using:

- RDP (assignTaxonomy), BLAST, and MapSeq
- A custom reference database combining MetaFishLib (fish) and MIDORI2 (non-fish)
- taxonomizr to summarise BLAST results

Currently using the standard Meta-Fish-Lib, with plans to incorporate a custom fish database when Jono completed.

## Set-up

Install packages, load required functions and get data.

In [0]:
source("Scripts/00_setup.R") #packages and functions

In [0]:
%sh
# cutadapt
pip install -q cutadapt
cutadapt --version

# vsearch install
apt-get install -y vsearch

# BLAST+ installed
apt-get -q update
apt-get -q install -y ncbi-blast+

In [0]:
%sh
which cutadapt
which vsearch

In [0]:
#temp solution
cutadapt_loc <-"/local_disk0/.ephemeral_nfs/envs/pythonEnv-bcdf6ce5-ce82-4cd8-9696-0ba1c3e7d2cc/bin/cutadapt"

In [0]:
test_data_name <- "Marchamley" # change here to update test data (same name as folder)
test_data_loc <- file.path("Data", "Raw", test_data_name)
results_loc = paste("Results/", test_data_name)
print(head(list.files(test_data_loc)))

In [0]:
manifest <- make_manifest(test_data_loc)
write.csv(manifest, file.path("Data", "Temp", test_data_name, "manifest.csv"), row.names = FALSE)
head(manifest)

## Removing Ns

In [0]:
source("Scripts/01_remove_Ns.R")

## Identify and remove primers

In [0]:
# Riaz 2011

FWD <- "ACTGGGATTAGATACCCC"
REV <- "TAGAACAGGCTCCTCTAG"

In [0]:
# these would be options in actual pipeline

minimum <- 60 #Minimum read length cutoff. Recommend >0
copies <- 2 #Number of copies of a primer to be removed as sometimes dulication can occur. Recommended minimum is 2

In [0]:
source("Scripts/02_primer_removal.R")

## Generate quality plots

In [0]:
source("Scripts/03_raw_quality_plots.R")

In the plots below:
- Grey-scale heatmap shows the frequency of each quality score along the read lengths - looking for over 30 ish. 
- Green line is median quality score. 
- Orange line are quartiles. 
- Red line at the bottom represent the proportion of reads of that particular length. 

Our reads on average are approx 100bp long after primers removed in the first two samples, which is about right for these primers.

In [0]:
print(plotQualityProfile(fnFs.cut[1:2]))

In [0]:
print(plotQualityProfile(fnRs.cut[1:2]))

## Cleaning the data (filterAndTrim)

Parameters for filterAndTrim include:
- **maxN** - after truncation, sequences with more then X Ns will be disgarded
- **truncQ** - Truncate reads at the first instance of a quality score less then or equal to X
- **rm.phix** - Discard reads that match against the phiX genome
- **maxEE** - After truncation, reads with higher than X expected errors will be discarded
- **minLen** - Remove reads with length less than 60 (note these should gave already been removed by cutadapt)
- **multithread** - input files are filtered in parallel (logical)
- **truncLen** - controls where each read is cut (truncated) based on its length

In [0]:
truncLen=c(80,95) # where the quality dropped in previous plots (RingTrial = 80 & 95)
maxEE <-  c(2,2) #maxEE value 
truncQ <- 2 #truncQ value
minLen <- 50 #Impose a minimum length cutoff

#other options in NEOF pipeline are subset (only run a portion of the data) and marker (for better labelling when running multiple markers)

In [0]:
source("Scripts/04_filterAndTrim.R")

In [0]:
plotQualityProfile(fnFs.filtN[5:6])

In [0]:
plotQualityProfile(fnRs.filtN[1:2])

## Generate error model

In [0]:
source("Scripts/05_generate_error_model.R")

In the plots below:
- Error rates for each possible transition (e.g. A->C, A->G) are shown
- Red line = expected based on quality score
- Black line = estimate 
- Black dots = observed 

We want black lines and black dots to match. We also want to see a rough negative correlation. 

If it looks weird, can increase the number of bases the function is using (default is 100 million).

In [0]:
plotErrors(errF, nominalQ = TRUE)


In [0]:
plotErrors(errR, nominalQ = TRUE)

Common to see 'hook' between 30 - 40 with NovaSeq and MiniSeq data when visualising in DADA2. Couple of forums have discussion on this topic. Not to be of too much concern unless seeing any weird results.

## Deplication, merging and chimera removal

### DADA2

In [0]:
source("Scripts/06a_derep_DADA2.R")

### VSEARCH de novo

Documentation for VSEARCH [here](https://torognes.github.io/vsearch/). Found bash code for VSEARCH here which I've adapted - https://github.com/mgdesaix/metabarcoding-pipeline. 

In [0]:
%sh
bash Scripts/06b_derep_VSEARCH.sh # must change data name in script

## Sequence tracking

In [0]:
source("Scripts/07a_sequence_tracking.R")

Input and filtered columns will be the same for both pipelines, as they were done in the filter and trim step. They are not repeated below.

Let's look at just the VSEARCH steps post filterAndTrim. They are in a different order the DADA2 (merging happens first in VSEARCH) so please take note of the column names.

In [0]:
%sh
bash Scripts/07b_sequence_tracking.sh # must change data name in script

The number of sequences should look roughly similar between the two methods.

We now need to get the data in a similar format to the DADA2 outputs.

In [0]:
source("Scripts/07c_format_VSEARCH.R")

Check out data outputs.

In [0]:
# load dataset
ASV_tab_DADA2 <- read.table(
  file = file.path("Data", "Processed", test_data_name, "06_ASV_counts_DADA2.tsv"),
  header = TRUE,
  sep = "\t",
  row.names = 1,
  check.names = FALSE
)

# inspect
display(head(ASV_tab_DADA2))
dim(ASV_tab_DADA2)

In [0]:
ASV_tab_VSEARCH <- read.table(
  file = file.path("Data", "Processed", test_data_name, "06b_ASV_counts_VSEARCH.tsv"),
  header = TRUE,
  sep = "\t",
  row.names = 1,
  check.names = FALSE
)

# inspect
display(head(ASV_tab_VSEARCH))
dim(ASV_tab_VSEARCH)

In [0]:
ASV_seq_DADA2 <- readDNAStringSet(file.path("Data", "Processed", test_data_name, "06_ASV_seqs_DADA2.fasta"))
ASV_seq_DADA2

In [0]:
ASV_seq_VSEARCH <- readDNAStringSet(file.path("Data", "Processed", test_data_name, "06b_ASV_seqs_VSEARCH.fasta"))
ASV_seq_VSEARCH

## Assign taxonomy

### RDP

In [0]:
source("Scripts/08a_assign_taxonomy_RDP.R") 

In [0]:
source("Scripts/08c_assign_taxonomy_RDP_VSEARCH.R")

### BLAST

Tidy MetaFishLib for BLAST.

In [0]:
%python
import pandas as pd
# Load your database
mfl_db = pd.read_csv("Data/Databases/Meta-fish-lib/references.12s.miya.cleaned.v268.csv")

# Clean sequences + names
mfl_db = mfl_db.dropna(subset=["gbAccession", "nucleotides"])
mfl_db["seq"] = mfl_db["nucleotides"].str.upper().str.replace(r"[^ACGT]", "", regex=True)
mfl_db["species"] = mfl_db["sciNameValid"].str.replace(r"[^A-Za-z ]", "", regex=True)

# Write FASTA
with open("Data/Databases/Meta-fish-lib/metafish.fasta", "w") as f:
    for _, row in mfl_db.iterrows():
        header = f">{row['gbAccession']} {row['species']}"
        f.write(header + "\n")
        f.write(row["seq"] + "\n")

In [0]:
%sh
# to make BLAST formatted database from curated Meta Fish Lb
makeblastdb -in /Workspace/Users/dina.simons@environment-agency.gov.uk/Fish_pipeline_comparisons/Data/Databases/Meta-fish-lib/metafish.fasta -dbtype nucl -out Data/Databases/12S_fish_db/12S_fish_db

Now run blast.

In [0]:
%sh
bash Scripts/08b_assign_taxonomy_BLAST.sh # must change data name in script

In [0]:
%sh
bash Scripts/08d_assign_taxonomy_BLAST_VSEARCH.sh # must change data name in script

Quick look at BLAST outputs.

In [0]:
# read MetaFishLib database
mfl_db <- read.csv(
  file = file.path(path, "Data", "Databases", "Meta-fish-lib", "references.12s.miya.cleaned.v268.csv"),
  stringsAsFactors = FALSE
)

# make taxonomy lookup table
taxonomy_lookup <- mfl_db %>%
  select(gbAccession, genus, family, order, class, phylum)

In [0]:
# set base paths
path <- getwd()
processed_dir <- file.path(path, "Data", "Processed", test_data_name)

## dada2 BLAST output
BLAST_DADA2_output <- read.table(
  file = file.path(processed_dir, "08b_ASVs_DADA2_blast_Metafishlib.txt"),
  sep = "\t",
  header = FALSE,
  stringsAsFactors = FALSE
)

# clean species column
BLAST_DADA2_output <- BLAST_DADA2_output %>%
  mutate(
    V3 = sub("^[^ ]+ ", "", V3),  # remove accession prefix
    V3 = gsub("_", " ", V3)       # optional: make names readable
  )

# set column names (BLAST format 6)
colnames(BLAST_DADA2_output) <- c(
  "qseqid", "sseqid", "species", "pident", "length",
  "mismatch", "gapopen", "qstart", "qend",
  "sstart", "send", "evalue", "bitscore"
)

# join taxonomy lookup
BLAST_DADA2_output <- BLAST_DADA2_output %>%
  left_join(taxonomy_lookup, by = c("sseqid" = "gbAccession"))

# inspect
display(BLAST_DADA2_output)


In [0]:
#set base paths
path <- getwd()
processed_dir <- file.path(path, "Data", "Processed", test_data_name)

## vsearch BLAST output
BLAST_VSEARCH_output <- read.table(
  file = file.path(processed_dir, "08b_ASVs_VSEARCH_blast_Metafishlib.txt"),
  sep = "\t",
  header = FALSE,
  stringsAsFactors = FALSE
)

# clean species column
BLAST_VSEARCH_output <- BLAST_VSEARCH_output %>%
  mutate(
    V3 = sub("^[^ ]+ ", "", V3),  # remove accession prefix
    V3 = gsub("_", " ", V3)       # optional: make names readable
  )

# set column names (BLAST outfmt 6)
colnames(BLAST_VSEARCH_output) <- c(
  "qseqid", "sseqid", "species", "pident", "length",
  "mismatch", "gapopen", "qstart", "qend",
  "sstart", "send", "evalue", "bitscore"
)

# join taxonomy lookup
BLAST_VSEARCH_output <- BLAST_VSEARCH_output %>%
  left_join(taxonomy_lookup, by = c("sseqid" = "gbAccession"))

# inspect
display(BLAST_VSEARCH_output)

#### Condense BLAST taxonomy

Condense taxonomy (lowest common ancestor) for BLAST outputs. Need to format the accessions database first though.

In [0]:
%sh
# directly downloaded larger accession dataset and uploaded to lab zone
#filtering to reduce size. created in sql in next script.

test_data_name="Windermere_2017"

cat Data/Processed/${test_data_name}/08b_ASVs_*_blast_Metafishlib.txt | cut -f2 | sort | uniq > Data/Databases/${test_data_name}_accessions.txt

zgrep -Ff Data/Databases/${test_data_name}_accessions.txt /Volumes/prd_dash_lab/ea_csg_research_edna_restricted/shared_external_volume/Fish_pipeline_comparison/Databases/nucl_gb.accession2taxid.gz > Data/Databases/${test_data_name}_subset.txt

In [0]:
# Takes a text file of accession → taxid mappings and converts it into a fast, searchable SQLite database

# convert subset to SQLite database
read.accession2taxid(
  taxaFiles = file.path("Data", "Databases", paste0(test_data_name, "_subset.txt")),
  sqlFile   = file.path("Data", "Databases", paste0(test_data_name, "_small_accessionTaxa.sql")),
  overwrite = TRUE
)

# store path
sql <- file.path("Data", "Databases", paste0(test_data_name, "_small_accessionTaxa.sql"))

Convert NCBI accession numbers to taxonomic IDs.

Working until here -------------------------------------------------------

In [0]:
dada2_accessions <- BLAST_DADA2_output$sseqid 
dada2_taxaId<-accessionToTaxa(dada2_accessions, sql)
print("Preview of taxa IDs for DADA2:")
head(dada2_taxaId)

In [0]:
VSEARCH_accessions <- BLAST_VSEARCH_output$sseqid 
VSEARCH_taxaId<-accessionToTaxa(VSEARCH_accessions, sql)
print("Preview of taxa IDs for VSEARCH:")
head(VSEARCH_taxaId)

In [0]:
taxa_DADA2 <- getTaxonomy(dada2_taxaId,sql)

In [0]:
source("Scripts/09_condensing_taxonomy.R")

### MapSeq

In [0]:
str(dada2_species)

## Jono pipeline

In [0]:
%skip
%python
%run Scripts/00_setup.py #packages and functions

In [0]:
%skip
%sh
pip install /Workspace/Shared/monitoring/EA_diatom_dada2_pipeline_DASH/packages/dokdo

In [0]:
%skip
%python
test_data_loc = "Data/Raw/RingTrial_Sean/"
results_loc = "Results"
os.listdir(test_data_loc)[:5]

In [0]:
%skip
%python
file_paths = get_files(results_loc)
samples = sort_paths(file_paths, results_loc)
export(samples, results_loc)

In [0]:
%skip
%sh
#define the analysis location (not retained from other cell)
results_loc = "/Results/"

qiime tools import \
    --type 'SampleData[PairedEndSequencesWithQuality]' \
    --input-path $results_loc/pe-33-manifest \
    --output-path /tmp/paired-end-demux.qza \
    --input-format PairedEndFastqManifestPhred33V2 && rm $results_loc/*fastq.gz

qiime demux summarize \
    --i-data /tmp/paired-end-demux.qza \
    --o-visualization /tmp/paired-end-demux.qzv

mv /tmp/paired-end-demux.qza $results_loc/paired-end-demux.qza
mv /tmp/paired-end-demux.qzv $results_loc/paired-end-demux.qzv